# 🎙️ AI Voice Clone Server (XTTS-v2)

**Setup (ONE TIME ONLY):**
1. Runtime → Change runtime type → **T4 GPU** select karo (free)
2. Cell 1 run karo (install — 3-5 min)
3. Cell 2 me apni awaaz ka sample upload karo (6-30 sec, clean audio)
4. Cell 3 run karo (model load — 1-2 min)
5. Cell 4 run karo — **URL milega**, wo app Settings → Voice Cloning me paste karo

**Har baar bas:** Cell 1, 3, 4 run karo. Sample Drive me save rahega.

⚠️ Notebook band mat karo jab tak video bana rahe ho. 90 min idle pe auto-band ho jayega.

In [ ]:
#@title Cell 1: Install Dependencies (3-5 min)
# coqui-tts = maintained fork of Coqui TTS (old TTS==0.22.0 fails on new Colab Python)
# transformers pinned to 4.44.2 — newer versions removed isin_mps_friendly and break coqui-tts
!pip install -q coqui-tts flask
!pip install -q transformers==4.44.2
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print("✅ Install complete!")

In [ ]:
#@title Cell 2: Voice Sample Upload (ONE TIME)
#@markdown Apni awaaz ka 6-30 second ka clean sample upload karo (mp3/wav/m4a).
#@markdown Background noise nahi hona chahiye. Ye sample Google Drive me save hoga.

import os
from google.colab import drive, files

drive.mount('/content/drive')
SAMPLE_DIR = '/content/drive/MyDrive/voice_clone'
os.makedirs(SAMPLE_DIR, exist_ok=True)

SAMPLE_PATH = os.path.join(SAMPLE_DIR, 'my_voice_sample.wav')

if os.path.exists(SAMPLE_PATH):
    print(f"✅ Sample already saved: {SAMPLE_PATH}")
    print("   (Naya sample chahiye to purana delete karke dobara upload karo)")
else:
    print("📤 Apna voice sample upload karo:")
    uploaded = files.upload()
    if uploaded:
        fname = list(uploaded.keys())[0]
        # Convert to 24kHz mono WAV for best XTTS quality
        !ffmpeg -y -i "$fname" -ar 24000 -ac 1 "$SAMPLE_PATH" 2>/dev/null
        os.remove(fname)
        dur = !ffprobe -v error -show_entries format=duration -of default=noprint_wrappers=1:nokey=1 "$SAMPLE_PATH"
        print(f"✅ Sample saved to Drive: {SAMPLE_PATH} ({float(dur[0]):.1f}s)")
    else:
        print("❌ Koi file upload nahi hui!")

In [ ]:
#@title Cell 3: Load XTTS-v2 Model (1-2 min)
import os, torch

# ── Compatibility shim: newer transformers removed isin_mps_friendly ──
import transformers.pytorch_utils as _pu
if not hasattr(_pu, "isin_mps_friendly"):
    _pu.isin_mps_friendly = lambda elements, test_elements: torch.isin(elements, test_elements)
    print("[patch] injected isin_mps_friendly shim")

from TTS.api import TTS

SAMPLE_PATH = '/content/drive/MyDrive/voice_clone/my_voice_sample.wav'
assert os.path.exists(SAMPLE_PATH), "❌ Pehle Cell 2 me sample upload karo!"

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cpu':
    print("⚠️ GPU nahi mila — voice generation 3-5x slow hogi")

print("Loading XTTS-v2 model (first time ~2GB download)...")
tts_model = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print("✅ Model loaded! Ready to clone your voice.")

In [ ]:
#@title Cell 4: Start Voice Server + Tunnel
#@markdown Ye cell server start karega aur ek **URL** dega.
#@markdown Wo URL app me **Settings → Voice Cloning** me paste karo.

import os, io, re, time, uuid, subprocess, threading
from flask import Flask, request, send_file, jsonify

SAMPLE_PATH = '/content/drive/MyDrive/voice_clone/my_voice_sample.wav'
PORT = 7860

app = Flask(__name__)

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'model': 'xtts_v2', 'sample': os.path.basename(SAMPLE_PATH)})

@app.route('/tts', methods=['POST'])
def generate():
    try:
        data = request.get_json(force=True)
        text = (data.get('text') or '').strip()
        lang = data.get('language', 'en')
        if not text:
            return jsonify({'error': 'No text provided'}), 400
        # Cap text length to avoid OOM on free tier
        text = text[:2000]
        out_path = f'/tmp/tts_{uuid.uuid4().hex}.wav'
        tts_model.tts_to_file(
            text=text,
            speaker_wav=SAMPLE_PATH,
            language=lang,
            file_path=out_path
        )
        return send_file(out_path, mimetype='audio/wav', as_attachment=False)
    except Exception as e:
        return jsonify({'error': str(e)}), 500

# Start Flask in background thread
server_thread = threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=PORT, debug=False, use_reloader=False),
    daemon=True
)
server_thread.start()
time.sleep(2)

# Start cloudflared tunnel
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

print("⏳ Tunnel starting...")
url_found = False
deadline = time.time() + 60
for line in tunnel_proc.stdout:
    if time.time() > deadline:
        break
    match = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', line)
    if match:
        url = match.group(1)
        print()
        print("=" * 60)
        print(f"🎙️ VOICE CLONE SERVER READY!")
        print(f"📋 Ye URL app me paste karo:")
        print()
        print(f"   {url}")
        print()
        print("App → Settings → Voice Cloning → URL paste → Test → Enable")
        print("=" * 60)
        url_found = True
        break

if not url_found:
    print("❌ Tunnel URL nahi mila. Cell dobara run karo.")
else:
    print("\n⚠️ Ye cell band mat karo jab tak voice generate karni hai.")
    print("⚠️ 90 min idle pe Colab auto-disconnect ho jayega.")
    # Keep cell alive
    try:
        while True:
            time.sleep(60)
    except KeyboardInterrupt:
        tunnel_proc.terminate()
        print("Server stopped.")